# 03 — Deterministic: Trend + Fourier Seasonality (Store Sales)

**Goal of this notebook:** build the first *learned* model — a `LinearRegression` on **deterministic** features (a linear trend plus weekly/annual **Fourier** seasonality) — and score it on the same 16-day validation set as the baseline.

"Deterministic" means every feature depends **only on the calendar date** (not on past sales), so each column is knowable in advance and extends straight into the future horizon with no leakage. This is the trend/seasonality machinery from the Kaggle Time Series course.

**What you'll see:** the deterministic model lands roughly at *parity* with the seasonal-naive baseline here — an honest, useful result. Seasonal-naive is a strong bar, and pure trend+seasonality (no lags, promotions, or holidays yet) doesn't beat it on this multi-series retail data. The measurable-margin win that SC-003 targets is expected from the later stages (features, then the hybrid).

Run it top-to-bottom from a fresh kernel.

## 0. Setup

Put the repo root on the import path so the reusable `src/` package is importable whether the kernel started in `notebooks/` or at the repo root. All loading, splitting, feature-building, scoring, and the model itself live in `src/` — the notebook stays thin and the logic is reused across stages.

In [1]:
import sys
from pathlib import Path

# The kernel's working directory is `notebooks/`, but our reusable code lives in the
# project's `src/` package one level up. Put the repo root on the import path so
# `import src.data` works whether the kernel started in the repo root or in notebooks/.
here = Path.cwd()
repo_root = here if (here / "src").exists() else here.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

import src.data as data
import src.features as features
import src.validation as validation
import src.models as models

print("repo root:", repo_root)

repo root: D:\AIO\M01\Conquer\TimeSeries


## 1. Load and reindex the data (gap-free)

Load `train.csv` and reindex every `(store_nbr, family)` series onto a complete daily calendar. Closed days (absent from the raw file) become `sales = 0`, so the per-series index is gap-free. This matters doubly for the deterministic model: the Fourier terms are computed from each row's calendar **position**, so a missing day would slide the weekly/annual waves out of phase. We re-assert the gap-free guarantee before building any seasonality feature.

In [2]:
raw = data.load_train()
df = data.reindex_series_gapfree(raw)
validation.assert_gapfree(df)

print(f"raw rows:       {len(raw):,}")
print(f"gap-free rows:  {len(df):,}")
print(f"date range:     {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"series:         {df[['store_nbr', 'family']].drop_duplicates().shape[0]:,}")

raw rows:       3,000,888
gap-free rows:  3,008,016
date range:     2013-01-01 -> 2017-08-15
series:         1,782


## 2. Build the validation split

Carve off the **last 16 days** as the time-based validation set (2017-07-31 → 2017-08-15) — the exact same window the baseline used. It mirrors the real forecast horizon and comes strictly *after* the training data, so the local RMSLE behaves like the leaderboard and no future day leaks into training.

In [3]:
train, val = validation.train_validation_split(df)

print(f"train:   {train['date'].min().date()} -> {train['date'].max().date()}  ({len(train):,} rows)")
print(f"validation: {val['date'].min().date()} -> {val['date'].max().date()}  ({len(val):,} rows)")
print(f"validation days: {val['date'].nunique()}")

train:   2013-01-01 -> 2017-07-30  (2,979,504 rows)
validation: 2017-07-31 -> 2017-08-15  (28,512 rows)
validation days: 16


## 3. The deterministic features (trend + Fourier)

`make_deterministic_features` wraps statsmodels' `DeterministicProcess` and `CalendarFourier` to produce, from a daily date index alone:

- **`const`** — an intercept (the per-series level is learned against this),
- **`trend`** — a linear counter `1, 2, 3, …` over time,
- **weekly Fourier** — 3 sin/cos harmonics for the 7-day cycle (the weekend rhythm),
- **annual Fourier** — 4 sin/cos harmonics for the ~365-day cycle (the December surge, summer dip).

The harmonic counts come from the EDA periodogram evidence (~2–3 weekly, ~3–5 annual). Below we build the table over the full **train ∪ horizon** index so the `trend` counter is continuous, then peek at the columns. Because the features are pure functions of the date, the horizon rows are computable today — that's the whole point.

In [4]:
# One continuous daily index covering training days plus the 16-day horizon.
train_dates = pd.DatetimeIndex(train['date'].sort_values().unique())
horizon = pd.date_range(
    train_dates.max() + pd.Timedelta(days=1), periods=validation.HORIZON_DAYS, freq="D"
)
X_demo = features.make_deterministic_features(train_dates.append(horizon))

print("feature matrix shape:", X_demo.shape)
print("columns:", list(X_demo.columns))
X_demo.tail(3)

feature matrix shape: (1688, 16)
columns: ['const', 'trend', 'sin(1,freq=W-SUN)', 'cos(1,freq=W-SUN)', 'sin(2,freq=W-SUN)', 'cos(2,freq=W-SUN)', 'sin(3,freq=W-SUN)', 'cos(3,freq=W-SUN)', 'sin(1,freq=YE-DEC)', 'cos(1,freq=YE-DEC)', 'sin(2,freq=YE-DEC)', 'cos(2,freq=YE-DEC)', 'sin(3,freq=YE-DEC)', 'cos(3,freq=YE-DEC)', 'sin(4,freq=YE-DEC)', 'cos(4,freq=YE-DEC)']


,const,trend,"sin(1,freq=W-SUN)","cos(1,freq=W-SUN)","sin(2,freq=W-SUN)","cos(2,freq=W-SUN)","sin(3,freq=W-SUN)","cos(3,freq=W-SUN)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)"
2017-08-13,1.0,1686.0,-0.781831,0.62349,-0.974928,-0.222521,-0.433884,-0.900969,-0.655156,-0.755493,0.989932,0.141540,-0.840618,0.541628,0.280231,-0.959933
2017-08-14,1.0,1687.0,0.000000,1.00000,0.000000,1.000000,0.000000,1.000000,-0.668064,-0.744104,0.994218,0.107381,-0.811539,0.584298,0.213521,-0.976938
2017-08-15,1.0,1688.0,0.781831,0.62349,0.974928,-0.222521,0.433884,-0.900969,-0.680773,-0.732494,0.997325,0.073095,-0.780296,0.625411,0.145799,-0.989314


## 4. Fit the deterministic model and predict the horizon

`DeterministicModel` fits a **separate `LinearRegression` per series** on the shared date features, in `log1p` space (then inverts with `expm1` and clips to `>= 0`). Each series gets its own level, trend slope, and seasonal amplitude.

Two data realities would make a naive fit *explode*, and the wrapper handles both — worth understanding, because they're typical of real retail panels:

1. **Leading-zero blocks.** Many series carry `sales = 0` from 2013-01-01 until the store actually opened. A single linear trend through `[dead zeros → active sales]` is a step the smooth basis can't represent, and least-squares *overshoots wildly* — predicting tens of thousands for a series whose max is a few thousand, even on its own training data. **Fix:** fit each series only on its **active history** (from its first non-zero day), keeping interior closed-day zeros (those are real signal).
2. **Short histories.** A store open less than a year can't identify an *annual* cycle — the yearly sin/cos columns barely vary over a few months, so their coefficients blow up and extrapolate insanely. **Fix:** drop the annual Fourier terms for series with under a year of active history; keep trend + weekly.

Series that never sold predict 0 (the natural near-zero fallback). The result is a tidy `[store_nbr, family, date, sales_pred]` frame — same shape as the baseline, so both score identically.

In [5]:
det_model = models.DeterministicModel().fit(train)
preds = det_model.predict(validation.HORIZON_DAYS)

print("prediction rows:", f"{len(preds):,}")
print("NaN predictions:", int(preds['sales_pred'].isna().sum()))
print(f"prediction range: {preds['sales_pred'].min():.2f} -> {preds['sales_pred'].max():,.0f}")
preds.head()

prediction rows: 28,512
NaN predictions: 0
prediction range: 0.00 -> 88,843


,store_nbr,family,date,sales_pred
0,1,AUTOMOTIVE,2017-07-31,4.513697
1,1,AUTOMOTIVE,2017-08-01,5.208059
2,1,AUTOMOTIVE,2017-08-02,4.679791
3,1,AUTOMOTIVE,2017-08-03,4.124665
4,1,AUTOMOTIVE,2017-08-04,4.916488


## 5. Score on the validation set (RMSLE, log space)

Merge the predictions onto the validation actuals by `(store_nbr, family, date)`, then compute **RMSLE** — the competition metric. `rmsle` takes *raw* sales and applies `log1p` internally, clipping predictions to `>= 0`. An exact-match merge with no missing rows confirms predictions and actuals are perfectly aligned.

In [6]:
scored = val.merge(preds, on=['store_nbr', 'family', 'date'], how='inner')
assert len(scored) == len(val), 'prediction/validation misalignment'
assert scored[['sales', 'sales_pred']].isna().sum().sum() == 0, 'missing values in scored frame'

det_rmsle = validation.rmsle(scored['sales'], scored['sales_pred'])
print(f"scored rows: {len(scored):,}")
print(f"deterministic validation RMSLE: {det_rmsle:.5f}")

scored rows: 28,512
deterministic validation RMSLE: 0.62188


## 6. Compare to the baseline

Recompute the seasonal-naive baseline on the same split and put the two numbers side by side. The deterministic model captures **smooth** trend and seasonality; the baseline simply *copies the last observed week*, which already encodes each series' recent level and exact weekly shape. On a 16-day horizon that's hard to beat with trend+seasonality alone.

In [7]:
base_preds = models.seasonal_naive_predict(train, validation.HORIZON_DAYS)
base_scored = val.merge(base_preds, on=['store_nbr', 'family', 'date'], how='inner')
baseline_rmsle = validation.rmsle(base_scored['sales'], base_scored['sales_pred'])

rel = (baseline_rmsle - det_rmsle) / baseline_rmsle * 100
print(f"baseline (seasonal-naive)  RMSLE: {baseline_rmsle:.5f}")
print(f"deterministic (trend+Fourier) RMSLE: {det_rmsle:.5f}")
print(f"relative change vs baseline: {rel:+.1f}%  (positive = improvement)")

baseline (seasonal-naive)  RMSLE: 0.61704
deterministic (trend+Fourier) RMSLE: 0.62188
relative change vs baseline: -0.8%  (positive = improvement)


## 7. Record the iteration

Append this score to `iteration_log.md` via `log_iteration`, which records the technique, its validation RMSLE, and the delta versus the best result so far. Logging the honest number — improvement *or not* — is the point: it makes each technique's real contribution traceable (FR-010).

> Note: re-running this cell appends another row — expected for an append-only log.

In [8]:
delta = validation.log_iteration(
    "deterministic: trend + weekly/annual Fourier (LinearRegression)",
    det_rmsle,
    notes="per-series fit on active history; annual dropped for <1yr series; log1p space",
)
print(f"logged deterministic RMSLE={det_rmsle:.5f}  delta={delta}")

logged deterministic RMSLE=0.62188  delta=+0.11197 (worse)


## Takeaway

The deterministic model (trend + weekly/annual Fourier) lands at a validation RMSLE of **~0.62 — essentially parity** with the seasonal-naive baseline (~0.617). That's an honest, informative outcome, not a failure:

- Smooth trend + Fourier seasonality captures the *shape* of demand but misses the **recent level shifts, promotions, holidays, and oil signal** that move sales day to day. The baseline's "copy last week" already banks the recent level for free.
- Building it surfaced two real data traps — **leading-zero blocks** and **short histories** — that we now handle robustly for every later stage that reuses these features.

SC-003 ("at least one improved model beats the baseline by a measurable margin, target ≥ 10%") is a **project-level** criterion — it stays open here and is expected to be met by the next stages, not by deterministic-only.

**Next (Stage 3 — `04_features.ipynb`):** add features one at a time on top of these deterministic terms — holiday indicators, calendar/payday/earthquake flags, promotions, forward-filled oil, and leak-safe lag/rolling features — re-scoring after each so the iteration log shows which technique earns the margin.